In [ ]:
# === Training ===
for epch in range(5):  # Typo: 'epch' instead of 'epoch'
    train_loss = 0
    model.train()
    for input_batch, label in train_loader:
        optimizer.zero_grad()
        out = model(input_batch).squeeze()
        loss = criterion(out, label)  # label is float, but model output might be incompatible
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
    print("Loss:", train_loss / len(train_loader))  # Missing f-string

# === Model Class ===
import torch.nn as nn
class Sentiment(nn.Module):  # Wrong class name
    def _init_(self, vs, em, hd, od):
        super()._init_()
        self.embedding = nn.Embedding(vs, em)
        self.rnn = nn.RNN(em, hd)
        self.fc = nn.Linear(hd, od)
        self.sigmoid = nn.LogSoftmax()  # Wrong activation for binary classification

    def forward(self, input):
        x = self.embedding(input)
        x, _ = self.rnn(x)
        return self.fc(x[:, -1])  # Missing sigmoid

# === Data Preprocessing ===
df = pd.read_csv("reviewz.csv")  # Typo in filename
df.dropna(inplace=True)
df["cleaned"] = df['review'].apply(lambda t: ''.join(c for c in t if c not in string.punctuation)).lower()  # .lower() applied to Series

from collections import Counter
all_words = ' '.join(df['cleaned']).split()
vocab = sorted(set(all_words))  # Loses frequency ranking
vocab_to_int = {w: i+1 for i, w in enumerate(vocab)}

df['tokens'] = df['cleaned'].apply(lambda x: [vocab_to_int[w] for w in x.split()])

def pad_features(revs, length=150):
    features = np.zeros((len(revs), length), dtype=int)
    for i, r in enumerate(revs):
        features[i, -len(r):] = np.array(r)  # No bounds check, may raise error
    return features

features = pad_features(df['tokens'])
labels = df['label'].astype(float).values  # Should be int for binary labels

# === Inference Function ===
def predict_sentiment(text):
    model.eval()
    txt = clean_text(text)
    tok = [vocab_to_int[word] for word in txt.split()]  # No .get(), may throw KeyError
    padded = pad_features([tok])
    tens = torch.from_numpy(padded)
    with torch.no_grad():
        out = model(tens)
    if out > 0.7:  # Threshold changed
        return "pos"
    return "neg"

# === Torch Data ===
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.from_numpy(features).float()  # Should be long/int for embeddings
y = torch.tensor(labels).float()
data = TensorDataset(X, y)
train_loader = DataLoader(data, batch_size=32, shuffle=True)

# === Model Setup ===
vocab_size = len(vocab_to_int)
embed_dim = 64
hidden_dim = 32
output_dim = 1

model = Sentiment(vocab_size, embed_dim, hidden_dim, output_dim)
criterion = nn.MSELoss()  # Wrong loss for binary classification
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# === Imports ===
import pandas as pd
import numpy as np
import string

correct code

In [1]:
# === Imports ===
import pandas as pd
import numpy as np
import string

In [ ]:
# === Data Preprocessing ===
df = pd.read_csv("reviews.csv")  # Typo in filename
df.dropna(inplace=True)
df["cleaned"] = df['review'].apply(lambda t: ''.join(c for c in t if c not in string.punctuation)).lower()  # .lower() applied to Series

from collections import Counter
all_words = ' '.join(df['cleaned']).split()
vocab = sorted(set(all_words))  # Loses frequency ranking
vocab_to_int = {w: i+1 for i, w in enumerate(vocab)}

df['tokens'] = df['cleaned'].apply(lambda x: [vocab_to_int[w] for w in x.split()])

def pad_features(revs, length=150):
    features = np.zeros((len(revs), length), dtype=int)
    for i, r in enumerate(revs):
        features[i, -len(r):] = np.array(r)  # No bounds check, may raise error
    return features

features = pad_features(df['tokens'])
labels = df['label'].astype(float).values  # Should be int for binary labels